# Todo

1. 先验证一下 Z3 推理机是否涉及常识；
2. 在每个步骤进行推理的时候，让模型输出完整的步骤；
3. 考虑 NLI 等模型

# 1. Z3 solver 在什么情况下才会 unsat

premise 1: A区在B区的西方；


premise 2: C区在B区的东方；

hypothesis: B区在C区西方；

In [1]:
from z3 import *
AreaSort, (A, B, C, D) = EnumSort('AreaSort', ['A_area', 'B_area', 'C_area', 'D_area'])
DirectionSort, (south, east, north, west)  = EnumSort('DirectionSort', ['south', 'east', 'north', 'west'])
Position = Function('position', AreaSort, AreaSort, DirectionSort)

In [2]:
s = Solver()

In [5]:
s.reset()
s.add([
    Position(C,B)==east, # premise 2
    Position(B,C)==west # hypothesis
])
print(s.check())

sat


In [6]:
s.reset()
s.add([
    Position(C,B)==east,
    Position(B,C)==north, # Error hypothesis: C区在B区的东方 --推导--> B区在C区北方 -> unsat
]) 
print(s.check())

sat


In [7]:
s.reset()
s.add([
    Position(C,B)==east,
    Not(Position(B,C)==west), # Not hypothesis
]) 
print(s.check())

sat


Z3 是一个“懒惰”的求解器，它只要能找到一种情况让所有的公式都为 True，它就返回 sat。

并且在我们没有给规则的情况下，它没有办法判断正误。

In [10]:
s.reset()

x = Const('x', AreaSort)
y = Const('y', AreaSort)
s.add(ForAll([x,y], Implies((Position(x, y) == east), (Position(y, x) == west)))) # 给定一个常识：如果x在y东方，y就在x的西方
s.add(Position(C,B)==east)
print(s.check())

s.add(Position(B,C)==north)
print(s.check())

# CYC

sat
unsat


在逻辑学中，$P \implies Q$ （如果 P 则 Q） 这个命题，只有在 P 为真 且 Q 为假 的时候才是假的（False），在其他所有情况下，它都是真（True）。


也就是说，只要 P 是假的，整个蕴含式就自动为真（这叫“实质蕴涵”）。


Z3 是一个“懒惰”的求解器，它只要能找到一种情况让所有的公式都为 True，它就返回 sat。也就是说，如果我们只给规则，不给赋值，那么它只要找到一种情况让 P 为假，那么就 sat 了，比如 $Position(C,B)\neq east$。但这些 commonsense 是 LLM 的强项，要么让 LLM 来判断，要么让LLM 来补这部分的规则。

#### TODO：使用 LLM 来补充缺失的规则、commonsense 

# 2. 让 LLM 在每个 step 列举所有的 premises

In [6]:
from openai import OpenAI
import requests

class LLM:
    def __init__(self, base_url="http://localhost:4869/v1", api_key="EMPTY"):
        self.base_url = base_url
        self.api_key = api_key
        self.client = OpenAI(
            base_url=self.base_url,
            api_key=self.api_key,
        )
        self.health_check()

    def generate(self, data, model='qwen2.5-3b', args=None):
        chat_response = self.client.chat.completions.create(
            model=model,
            messages=[
                {"role": "user", "content": data},
            ],
            max_tokens=args['max_tokens'],
            temperature=args['temperature'],
            top_p=args['top_p'],
        )
        return chat_response.choices[0].message.content
    
    def health_check(self):
        health_url = self.base_url.replace("/v1", "") + "/health"
        response = requests.get(health_url)
        if response.status_code == 200:
            print("LLM API is available.")
        else:
            print(f"LLM API health check failed with status code: {response.status_code}")

In [7]:
llm = LLM()

LLM API is available.


In [8]:
args = {
    "max_tokens": 4096,
    "temperature": 0.1,
    "top_p": 0.8,
}

In [9]:
prompt_template = r"""
You are a logical reasoning expert. Your task is to analyze the given problem step by step, and output your reasoning in a structured format.

## INSTRUCTIONS
1. **Reasoning Process**: Think through the problem step by step. Each step should be a clear logical progression.
2. **Output Format**: Each reasoning step must be enclosed in `<step>` and `</step>` tags.
3. **Step Structure**: Within each step, use `<premise>` and `</premise>` tags for individual premises, and `<conclusion>` and `</conclusion>` tags for the conclusion of that step.
4. **Step Separation**: Each complete step block must be separated by exactly TWO newlines (\n\n).
5. **Final Answer**: At the end, after all steps, provide the correct answer index within `\boxed{}`.

## OUTPUT FORMAT SPECIFICATION
Your output should have this exact structure:

<step>
<premise>Your first premise or observation for this step</premise>
<premise>Your second premise (if applicable for this step)</premise>
<conclusion>The logical conclusion drawn from these premises in this step</conclusion>
</step>

<step>
<premise>First premise of the second step</premise>
<conclusion>Conclusion of the second step</conclusion>
</step>

[More steps as needed...]

\boxed{answer_index}

## TAG RULES
- **Each complete reasoning unit** must be wrapped in `<step>` and `</step>` tags
- **Premises** within a step go in `<premise>` and `</premise>` tags
- **Conclusion** of a step goes in `<conclusion>` and `</conclusion>` tags
- Each step must have at least one `<premise>` and one `<conclusion>`
- You can have multiple `<premise>` tags within a single step
- The final conclusion of the entire reasoning chain should be in the last step's `<conclusion>`

## STEP SEPARATION RULES
- Step 1: `<step> ... </step>`
- [BLANK LINE]
- Step 2: `<step> ... </step>`
- [BLANK LINE]
- Step 3: `<step> ... </step>`
- etc.

## EXAMPLES

### Example 1: Simple Logic
**Input**: "All cats are animals. Fluffy is a cat. What is Fluffy?"
**Output**:
<step>
<premise>All cats are animals.</premise>
<premise>Fluffy is a cat.</premise>
<conclusion>Therefore, Fluffy is an animal.</conclusion>
</step>

\boxed{animal}

### Example 2: Mathematical Reasoning
**Input**: "What is 3 × 4? (A) 7 (B) 12 (C) 34"
**Output**:
<step>
<premise>Multiplication is repeated addition: 3 × 4 = 3 + 3 + 3 + 3</premise>
<conclusion>3 + 3 + 3 + 3 = 12</conclusion>
</step>

<step>
<premise>The calculation yields 12.</premise>
<premise>Option (B) is 12.</premise>
<conclusion>Therefore, the correct answer is option (B).</conclusion>
</step>

\boxed{B}

### Example 3: Conditional Reasoning
**Input**: "If it is raining, then I carry an umbrella. I am not carrying an umbrella. Is it raining?"
**Output**:
<step>
<premise>Conditional statement: If it is raining → I carry an umbrella</premise>
<conclusion>This establishes a logical relationship between rain and umbrella.</conclusion>
</step>

<step>
<premise>The contrapositive: If I am not carrying an umbrella → it is not raining</premise>
<conclusion>This is the logically equivalent form of the conditional.</conclusion>
</step>

<step>
<premise>I am not carrying an umbrella (given fact).</premise>
<premise>Applying the contrapositive: If not carrying umbrella → not raining</premise>
<conclusion>Therefore, it is not raining.</conclusion>
</step>

\boxed{No}

## COMMON MISTAKES TO AVOID
1. ❌ DON'T combine multiple inferences in one step
2. ❌ DON'T skip showing premises
3. ❌ DON'T forget to wrap premises and conclusions in tags
4. ❌ DON'T put reasoning after the \boxed{} answer
5. ❌ DON'T use markdown or other formatting inside the tags

## SPECIAL CASES
- If the problem has no choices, output only the reasoning steps
- If a step has multiple premises, list them in separate `<premise>` tags
- If you need to make an assumption, state it explicitly in a premise
- If the problem is mathematical, show calculations clearly

Now, solve the following problem.
"""

John 在以下六门课程中各获得一个成绩：经济学、地质学、历史学、意大利语、物理学和俄语。按从高到低的顺序，可能的成绩等级分别为 A、B、C、D 和 E。其中，E 是唯一的不及格成绩。当且仅当两个字母成绩在字母表中相邻时，它们才被称为是“连续的”。John 在地质学和物理学上的成绩是连续的。他在意大利语和俄语上的成绩也是连续的。他在经济学上获得的成绩高于历史学。他在地质学上获得的成绩高于物理学。"


假设 John 在物理学上的成绩高于他在意大利语上的成绩，且两者是连续的；同时，他在俄语和物理学上的成绩互不相同。请问以下哪项陈述必然是正确的？


(A) John 同时获得了 A 和 B 这两个成绩。


(B) John 同时获得了 A 和 C 这两个成绩。


(C) John 同时获得了 B 和 D 这两个成绩。


(D) John 同时获得了 B 和 E 这两个成绩。


(E) John 同时获得了 D 和 E 这两个成绩。


正确答案: C

In [10]:
context = "John receives one grade for each of the following six courses: economics, geology, history, Italian, physics, and Russian. From highest to lowest, the possible grades are A, B, C, D, and E. E is the only failing grade. Two letter grades are consecutive if and only if they are adjacent in the alphabet. John's grades in geology and physics are consecutive. His grades in Italian and Russian are consecutive. He receives a higher grade in economics than in history. He receives a higher grade in geology than in physics."
question = "Assume that John's grade in physics is higher than his grade in Italian and consecutive with it and that his grades in Russian and physics differ. Which one of the following must be true?"
options = "(A)John receives both an A and a B.\n\n (B)John receives both an A and a C. \n\n (C)John receives both a B and a D. \n\n (D)John receives both a B and an E. \n\n (E) John receives both a D and an E."
gt = 'C'

In [11]:
prompt = prompt_template + context + question + options
response = llm.generate(data=prompt, args=args)

In [12]:
from pprint import pprint
pprint(response)

('<step>\n'
 "<premise>John's grades in geology and physics are consecutive.</premise>\n"
 "<premise>John's grades in Italian and Russian are consecutive.</premise>\n"
 '<premise>John receives a higher grade in economics than in '
 'history.</premise>\n'
 '<premise>John receives a higher grade in geology than in physics.</premise>\n'
 "<premise>John's grade in physics is higher than his grade in Italian and "
 'consecutive with it.</premise>\n'
 "<premise>John's grades in Russian and physics differ.</premise>\n"
 '<conclusion>From the information provided, we know the '
 'following:</conclusion>\n'
 '</step>\n'
 '\n'
 '<step>\n'
 "<premise>Since John's grade in physics is higher than his grade in Italian "
 'and consecutive with it, the possible pairs for Italian and Physics are (B, '
 'C), (C, D), or (D, E).</premise>\n'
 "<conclusion>However, since John's grade in Russian and physics differ, the "
 'pair (D, E) is not possible because it would mean Russian is E, which is the '
 'only

### 尝试让LLM输出冗余 premises

假设通过规则/LLM先整理所有的premises，之前我写的方案也有这个预处理的步骤，大概就是将 premises 格式化为 \<p1>\</p1> \<p2>\</p2>

In [35]:
prompt_template = r"""
You are a logical reasoning expert. Your task is to analyze the given problem step by step, and output your reasoning in a structured format.

## INSTRUCTIONS
1. **Reasoning Process**: Think through the problem step by step. Each step should be a clear logical progression.
2. **Context Tracking (NEW RULE)**:
   In every step, you must include a `<context>` block that lists ALL currently available premises.
   The context must include:
   - All premises given in the problem
   - All conclusions derived in previous steps
   - Any assumption explicitly introduced earlier
3. **Output Format**: Each reasoning step must be enclosed in `<step>` and `</step>` tags.
4. **Step Structure**: Within each step, use:
   - `<context>` for the set of available premises
   - `<premise>` for the premises used in this step
   - `<conclusion>` for the conclusion of this step
5. **Step Separation**: Each complete step block must be separated by exactly TWO newlines (\n\n).
6. **Final Answer**: At the end, after all steps, provide the correct answer index within `\boxed{}`.

## OUTPUT FORMAT SPECIFICATION

Your output should have this exact structure:

<step>
<context>
All currently known premises and derived conclusions
</context>
<premise>Premise used in this step</premise>
<premise>Another premise used in this step</premise>
<conclusion>Conclusion of this step</conclusion>
</step>

<step>
<context>
Updated set of all known premises including previous conclusions
</context>
<premise>Premise used in this step</premise>
<conclusion>Conclusion of this step</conclusion>
</step>

[More steps as needed...]

\boxed{answer_index}


## TAG RULES

- Each reasoning unit must be wrapped in `<step>` `</step>`
- Each step must contain EXACTLY ONE `<context>` block
- `<context>` must list ALL currently available premises
- `<premise>` contains only the premises used in THIS step
- `<conclusion>` contains only the new inference of THIS step
- Each step must have at least:
  - one `<context>`
  - one `<premise>`
  - one `<conclusion>`
- The last step must contain the final logical conclusion


## CONTEXT RULES (IMPORTANT)

For step 1:
context = all premises from the problem

For step N:
context =  
- all original premises  
- all previous conclusions  
- all assumptions introduced earlier  

The context must grow monotonically.


## STEP SEPARATION RULES

- Step 1
<step>...</step>

(blank line)

- Step 2
<step>...</step>

(blank line)

- Step 3
<step>...</step>


## EXAMPLE

Input:
"All cats are animals. Fluffy is a cat. What is Fluffy?"

Output:

<step>
<context>
All cats are animals
Fluffy is a cat
</context>
<premise>All cats are animals</premise>
<premise>Fluffy is a cat</premise>
<conclusion>Fluffy is an animal</conclusion>
</step>

<step>
<context>
All cats are animals
Fluffy is a cat
Fluffy is an animal
</context>
<premise>Fluffy is an animal</premise>
<conclusion>Therefore the answer is animal</conclusion>
</step>

\boxed{animal}


## COMMON MISTAKES TO AVOID

1. ❌ Missing `<context>`
2. ❌ Context not including previous conclusions
3. ❌ Using a premise not listed in context
4. ❌ Multiple conclusions in one step
5. ❌ Putting reasoning after \boxed{}
6. ❌ Context shrinking instead of growing


## SPECIAL CASES

- If no choices → output only steps
- If assumption needed → add it to context in next step
- If math → show calculation in premise


Now, solve the following problem.
"""

In [36]:
prompt = prompt_template + context + question + options
response = llm.generate(data=prompt, args=args)

In [38]:
pprint(response)

("To solve this problem, let's break down the information provided and deduce "
 'the possible grades step by step.\n'
 '\n'
 '<step>\n'
 '<context>\n'
 'John receives one grade for each of the following six courses: economics, '
 'geology, history, Italian, physics, and Russian.\n'
 'From highest to lowest, the possible grades are A, B, C, D, and E. E is the '
 'only failing grade.\n'
 'Two letter grades are consecutive if and only if they are adjacent in the '
 'alphabet.\n'
 "John's grades in geology and physics are consecutive.\n"
 'His grades in Italian and Russian are consecutive.\n'
 'He receives a higher grade in economics than in history.\n'
 'He receives a higher grade in geology than in physics.\n'
 "John's grade in physics is higher than his grade in Italian and consecutive "
 'with it.\n'
 'His grades in Russian and physics differ.\n'
 '</context>\n'
 "<premise>John's grades in geology and physics are consecutive.</premise>\n"
 "<premise>John's grades in Italian and Russia

### 版本2

In [13]:
prompt_template = r"""
You are a logical reasoning expert. Your task is to analyze the given problem step by step, and output your reasoning in a structured format.

## INSTRUCTIONS
1. **Reasoning Process**: Think through the problem step by step. Each step should be a clear logical progression.
2. **Premise List**: In every step, you must include a `<premise_list>` block that lists ALL currently available premises.
    The premise list include:
   - All premises given in the problem
   - All conclusions derived in previous steps
   - Any assumption explicitly introduced earlier
3. **Output Format**: Each reasoning step must be enclosed in `<step>` and `</step>` tags.
4. **Step Structure**: Within each step, use:
   - `<premise_list>` for the set of available premises
   - `<premise>` for the premises used in this step
   - `<conclusion>` for the conclusion of this step
5. **Step Separation**: Each complete step block must be separated by exactly TWO newlines (\n\n).
6. **Final Answer**: At the end, after all steps, provide the correct answer index within `\boxed{}`.

## OUTPUT FORMAT SPECIFICATION
Your output should have this exact structure:
<step>
<premise_list>
    <p1> premises 1 given in the problem </p1>
    <p2> Premises 2 given in the problem </p2>
    ...
    <c1> The derived intermediate conclusion 1 from previous steps</c1>
    <c2> The derived intermediate conclusion 2 from previous steps</c2>
    ...
</premise_list>
<premise>Premise used in this step</premise>
<premise>Another premise used in this step</premise>
<conclusion>Conclusion of this step</conclusion>
</step>

<step>
<premise_list>
    <p1> premises 1 given in the problem </p1>
    <p2> Premises 2 given in the problem </p2>
    ...
    <c1> The derived intermediate conclusion 1 from previous steps</c1>
    <c2> The derived intermediate conclusion 2 from previous steps</c2>
    ...
</premise_list>
<premise>Premise used in this step</premise>
<conclusion>Conclusion of this step</conclusion>
</step>

[More steps as needed...]

\boxed{answer_index}


## TAG RULES
- Each reasoning unit must be wrapped in `<step>` `</step>`
- Each step must contain EXACTLY ONE `<premise_list>` block, ONE '<conclusion>` block and at less one `<premise>` block
- `<premise_list>` must list ALL currently available premises
- `<premise>` contains only the premises used in THIS step
- `<conclusion>` contains only the new inference of THIS step
- Each step must have at least:
  - one `<premise_list>`
  - one `<premise>`
  - one `<conclusion>`
- The last step must contain the final logical conclusion


## PREMISE LIST RULES (IMPORTANT)

For step 1:
premise_list = all premises from the problem

For step N:
premise_list =  
- all original premises  
- all previous conclusions  

The premise_list must grow monotonically.


## STEP SEPARATION RULES

- Step 1
<step>...</step>

(blank line)

- Step 2
<step>...</step>

(blank line)

- Step 3
<step>...</step>


## EXAMPLE

Input:
"<p1>All cats are animals.</p1> \n\n <p2> Fluffy is a cat. What is Fluffy? </p2>"

Output:

<step>
<premise_list>
<p1>All cats are animals</p1>
<p2>Fluffy is a cat</p2>
</premise_list>
<premise>All cats are animals</premise>
<premise>Fluffy is a cat</premise>
<conclusion>Fluffy is an animal</conclusion>
</step>

<step>
<premise_list>
<p1>All cats are animals</p1>
<p2>Fluffy is a cat</p2>
<c1>Fluffy is an animal</c1>
</premise_list>
<premise>Fluffy is an animal</premise>
<conclusion>Therefore the answer is animal</conclusion>
</step>

\boxed{animal}


## COMMON MISTAKES TO AVOID

1. ❌ DO NOT Missing `<premise_list>`;
2. ❗️ Premise list must include previous conclusions;
3. ❌ DO NOT USE a premise not listed in premise list;
4. ❌ DO NOT include Multiple conclusions in one step;
5. ❌ DO NOT PUT reasoning after \boxed{}
6. ❌ Context can not be shrinking instead of growing


## SPECIAL CASES

- If no choices → output only steps
- If assumption needed → add it to context in next step
- If math → show calculation in premise


Now, solve the following problem.
"""

In [14]:
context = """
<p1>John receives one grade for each of the following six courses: economics, geology, history, Italian, physics, and Russian.</p1>
<p2>From highest to lowest, the possible grades are A, B, C, D, and E. E is the only failing grade.</p2>
<p3>Two letter grades are consecutive if and only if they are adjacent in the alphabet. </p3>
<p4>John's grades in geology and physics are consecutive. His grades in Italian and Russian are consecutive.</p4>
<p5>He receives a higher grade in economics than in history. </p5> 
<p6> He receives a higher grade in geology than in physics. </p6>
<p7> Assume that John's grade in physics is higher than his grade in Italian and consecutive with it and that his grades in Russian and physics differ</p7>
"""
hypothesis = """
<h1> (A)John receives both an A and a B. </h1>
<h2> (B)John receives both an A and a C. </h2>
<h3> (C)John receives both a B and a D. </h3>
<h4> (D)John receives both a B and a D. </h4>
<h5> (E) John receives both a D and an E. </h5>
"""
question = ". Which one of the following must be true?"
options = "(A)John receives both an A and a B.\n\n (B)John receives both an A and a C. \n\n (C)John receives both a B and a D. \n\n (D)John receives both a B and a D. \n\n (E) John receives both a D and an E."
gt = 'C'

In [15]:
prompt = prompt_template + context + question + hypothesis
response = llm.generate(data=prompt, args=args)

In [16]:
pprint(response)

('<step>\n'
 '<premise_list>\n'
 '    <p1>John receives one grade for each of the following six courses: '
 'economics, geology, history, Italian, physics, and Russian.</p1>\n'
 '    <p2>From highest to lowest, the possible grades are A, B, C, D, and E. E '
 'is the only failing grade.</p2>\n'
 '    <p3>Two letter grades are consecutive if and only if they are adjacent '
 'in the alphabet.</p3>\n'
 "    <p4>John's grades in geology and physics are consecutive. His grades in "
 'Italian and Russian are consecutive.</p4>\n'
 '    <p5>He receives a higher grade in economics than in history.</p5> \n'
 '    <p6>He receives a higher grade in geology than in physics.</p6>\n'
 "    <p7>Assume that John's grade in physics is higher than his grade in "
 'Italian and consecutive with it and that his grades in Russian and physics '
 'differ</p7>\n'
 '</premise_list>\n'
 "<premise>John's grades in geology and physics are consecutive.</premise>\n"
 "<premise>John's grades in Italian and Russian are 

# 3 Translation Pipline

### Step 1: Coreference Resolution

In [17]:
coref_template = """
You are an expert in the field of Natural Language Processing. Given a piece of text, your task is to perform coreference resolution on the text, replacing all pronouns in the text with specific entities.
Here are an example:


## Input Text

Alice and Bob went to the beach. They were excited to see the waves and the sun. After some time, Alice said to Bob, "Let's build a sandcastle!" He agreed and started gathering some sand. While they were working, a seagull swooped down and tried to steal their snacks.


## Output

Alice and Bob went to the beach. Alice and Bob were excited to see the waves and the sun. After some time, Alice said to Bob, "Let's build a sandcastle!" Bob agreed and started gathering some sand. While Alice and Bob were working, a seagull swooped down and tried to steal Alice and Bob's snacks.
---

Now, please perform Coreference Resolution on the following paragraph:


## Input Text
${text}

## Output


"""
from string import Template
t = Template(coref_template)

In [18]:
text = """
<p1>John receives one grade for each of the following six courses: economics, geology, history, Italian, physics, and Russian.</p1>
<p2>From highest to lowest, the possible grades are A, B, C, D, and E. E is the only failing grade.</p2>
<p3>Two letter grades are consecutive if and only if they are adjacent in the alphabet. </p3>
<p4>John's grades in geology and physics are consecutive. His grades in Italian and Russian are consecutive.</p4>
<p5>He receives a higher grade in economics than in history. </p5> 
<p6> He receives a higher grade in geology than in physics. </p6>
<p7> Assume that John's grade in physics is higher than his grade in Italian and consecutive with it and that his grades in Russian and physics differ</p7>
"""

coref_prompt = t.safe_substitute(text=text)

In [19]:
coref_prompt

'\nYou are an expert in the field of Natural Language Processing. Given a piece of text, your task is to perform coreference resolution on the text, replacing all pronouns in the text with specific entities.\nHere are an example:\n\n\n## Input Text\n\nAlice and Bob went to the beach. They were excited to see the waves and the sun. After some time, Alice said to Bob, "Let\'s build a sandcastle!" He agreed and started gathering some sand. While they were working, a seagull swooped down and tried to steal their snacks.\n\n\n## Output\n\nAlice and Bob went to the beach. Alice and Bob were excited to see the waves and the sun. After some time, Alice said to Bob, "Let\'s build a sandcastle!" Bob agreed and started gathering some sand. While Alice and Bob were working, a seagull swooped down and tried to steal Alice and Bob\'s snacks.\n---\n\nNow, please perform Coreference Resolution on the following paragraph:\n\n\n## Input Text\n\n<p1>John receives one grade for each of the following six c

In [30]:
coref_response = llm.generate(data=coref_prompt, args=args)

In [31]:
pprint(coref_response)

('John receives one grade for each of the following six courses: economics, '
 'geology, history, Italian, physics, and Russian.\n'
 '\n'
 'From highest to lowest, the possible grades are A, B, C, D, and E. E is the '
 'only failing grade.\n'
 '\n'
 'Two letter grades are consecutive if and only if they are adjacent in the '
 'alphabet.\n'
 '\n'
 "John's grades in geology and physics are consecutive. John's grades in "
 'Italian and Russian are consecutive.\n'
 '\n'
 'He receives a higher grade in economics than in history.\n'
 '\n'
 'He receives a higher grade in geology than in physics.\n'
 '\n'
 "Assume that John's grade in physics is higher than his grade in Italian and "
 'consecutive with it and that his grades in Russian and physics differ.')


In [26]:
pattern = r"<p\d+>(.*?)</p\d+>"
# 使用 re.findall 找到所有匹配项
# re.DOTALL 模式让 . 可以匹配换行符，防止内容里有换行导致匹配失败
premises_list = re.findall(pattern, response, re.DOTALL)

for i, content in enumerate(premises_list, 1):
    # strip() 去掉内容前后的多余空格
    clean_content = content.strip()
    print("-"*20)
    print(clean_content)

### Step 2: Atom Decomposition

In [214]:
atomic_template = """
You are an expert in symbolic/logical reasoning, linguistic and natural language inference. You are given a sentence and its logical information. Generate a list of atomic facts that are strictly logically entailed from the given sentence.

## Instructions
1. Keep each fact independent and self-contained.

2. Each fact should make sense when read on its own.

3. Only write facts that are directly described or supported by the sentence.

4. The atomic facts must be logically entailed from the given sentence.

5. The atomic facts must be seperated by \n\n;

6. The response must follow this format:
- Atom 1: <The atom sentence 1>\n\n
- Atom 2: <The atom sentence 2>\n\n
...
- Atom N: <The atom sentence N>\n\n


## Example
Here are some examples:

** Provided Sentence **: Professional actors are in a summer performance. 

** Answer ** :

Atom 1: The people are professional.

Atom 2: The people are actors.

Atom 3: The performance is during summer. Task:

** Provided Sentence ** : ${sentence}

** Answer **:
"""

In [215]:
def extract_atom(atomic_template , sentence, llm, args):
    atom_prompt = atomic_template.format(sentence=sentence)
    res = llm.generate(data=atom_prompt, args=args)
    return res

In [76]:
sentence_test = "John receives one grade for each of the following six courses: economics, geology, history, Italian, physics, and Russian."
atom_res = extract_atom(atomic_template, sentence_test, llm, args)
atom_res

'Atom 1: John receives a grade for economics.\nAtom 2: John receives a grade for geology.\nAtom 3: John receives a grade for history.\nAtom 4: John receives a grade for Italian.\nAtom 5: John receives a grade for physics.\nAtom 6: John receives a grade for Russian.'

In [86]:
question_atom_list = []
for i, content in enumerate(premises_list, 1):
    clean_content = content.strip()
    # print(clean_content)
    atom_sent = extract_atom(atomic_template, clean_content, llm, args)
    print(atom_sent)
    attern = r"Atom\s*\d+\s*[:：]\s*(.*)"
    matches = re.findall(pattern, atom_sent)
    print(matches)
    print("------------------------")
    question_atom_list.extend(matches)

Atom 1: John receives a grade for economics.
Atom 2: John receives a grade for geology.
Atom 3: John receives a grade for history.
Atom 4: John receives a grade for Italian.
Atom 5: John receives a grade for physics.
Atom 6: John receives a grade for Russian.
[]
------------------------
Atom 1: The possible grades are A, B, C, D, and E.
Atom 2: E is a failing grade.
Atom 3: There are five possible grades in total (A, B, C, D, and E).
Atom 4: No grade other than E is considered failing.
[]
------------------------
** Answer **:

Atom 1: Two letter grades are consecutive if they are adjacent in the alphabet.
[]
------------------------
Based on the provided sentence, here are the atomic facts that are strictly logically entailed:

Atom 1: John's grades in geology are consecutive.
Atom 2: John's grades in physics are consecutive.
Atom 3: John's grades in Italian are consecutive.
Atom 4: John's grades in Russian are consecutive.
[]
------------------------
** Answer **:

Atom 1: John recei

In [92]:
import re

pattern = r"Atom\s*\d+\s*[:：]\s*(.*)"

question_atom_list = []

for i, content in enumerate(premises_list, 1):
    clean_content = content.strip()
    atom_sent = extract_atom(atomic_template, clean_content, llm, args)
    print(f"--- 处理第 {i} 条数据 ---")
    # print(atom_sent) # 调试时可以打印
    matches = re.findall(pattern, atom_sent)
    clean_matches = [m.strip() for m in matches]
    print(f"提取结果: {clean_matches}")
    print("------------------------")
    question_atom_list.extend(clean_matches)

print(f"CAN: 总共提取到 {len(question_atom_list)} 条原子事实。")


--- 处理第 1 条数据 ---
提取结果: ['John receives a grade for economics.', 'John receives a grade for geology.', 'John receives a grade for history.', 'John receives a grade for Italian.', 'John receives a grade for physics.', 'John receives a grade for Russian.']
------------------------
--- 处理第 2 条数据 ---
提取结果: ['The possible grades are A, B, C, D, and E.', 'E is a failing grade.', 'A is not a failing grade.', 'B is not a failing grade.', 'C is not a failing grade.', 'D is not a failing grade.']
------------------------
--- 处理第 3 条数据 ---
提取结果: ['Two letter grades are consecutive if they are adjacent in the alphabet.']
------------------------
--- 处理第 4 条数据 ---
提取结果: ["John's grade in geology is followed by his grade in physics.", "John's grade in physics is preceded by his grade in geology.", "John's grade in Italian is followed by his grade in Russian.", "John's grade in Russian is preceded by his grade in Italian."]
------------------------
--- 处理第 5 条数据 ---
提取结果: ['John receives a grade in e

In [93]:
len(question_atom_list)

26

### Step 3: Declarations Extraction

In [27]:
declaration_prompt = """
To accurately translate the context into a first-order logical expression, please analyze and extract the Declaration from the context. Please refer to the following example:

---

## Input
Four boys—Fred, Juan, Marc, and Paul—and three girls—Nita, Rachel, and Trisha—will be assigned to a row of five adjacent lockers, numbered consecutively 1 through 5, arranged along a straight wall. The following conditions govern the assignment of lockers to the seven children: Each locker must be assigned to either one or two children, and each child must be assigned to exactly one locker. Each shared locker must be assigned to one girl and one boy. Juan must share a locker, but Rachel cannot share a locker. Nita's locker cannot be adjacent to Trisha's locker. Fred must be assigned to locker 3.",

## Output
children = EnumSort([Fred, Juan, Marc, Paul, Nita, Rachel, Trisha])
lockers = EnumSort([1, 2, 3, 4, 5])
assigned = Function([children] -> [lockers])

---
## Input
A music store carries exactly ten types of CDs—both new and used of each of jazz, opera, pop, rap, and soul. The store is having a sale on some of these types of CDs. The following conditions must apply: Used pop is on sale; new opera is not. If both types of pop are on sale, then all soul is. If both types of jazz are on sale, then no rap is. If neither type of jazz is on sale, then new pop is. If either type of rap is on sale, then no soul is.

## Output
types = EnumSort([new_jazz, used_jazz, new_opera, used_opera, new_pop, used_pop, new_rap, used_rap, new_soul, used_soul])
on_sale = Function([types] -> [bool])

---
Now, please extract the Declaration from the following problem  and put the Declaration into ```python``` block :

## Input
${context}

"""

In [28]:
def declaration_ext(declaration_prompt, text):
    from string import Template
    declaration_T = Template(declaration_prompt)
    declaration_prompt = declaration_T.safe_substitute(context = text)
    declarations = llm.generate(data=declaration_prompt,args=args)
    pattern = r"```python\s+(.*?)```"
    matches = re.findall(pattern, declarations, re.DOTALL)
    return matches[-1]

In [29]:
from string import Template
declaration_T = Template(declaration_prompt)
context = " ".join(question_atom_list)

NameError: name 'question_atom_list' is not defined

In [155]:
declaration_prompt = declaration_T.safe_substitute(context = context)

In [156]:
declarations = llm.generate(data=declaration_prompt,args=args)

In [157]:
declarations

"```python\ngrades = EnumSort(['A', 'B', 'C', 'D', 'E'])\nconsecutive_grades = Function([grades] -> [bool])\njohn_grades = Function([['economics', 'geology', 'history', 'Italian', 'physics', 'Russian']] -> [grades])\n\n# Conditions\njohn_has_grade_economics = john_grades['economics'] != 'E'\njohn_has_grade_geology = john_grades['geology'] != 'E'\njohn_has_grade_history = john_grades['history'] != 'E'\njohn_has_grade_Italian = john_grades['Italian'] != 'E'\njohn_has_grade_physics = john_grades['physics'] != 'E'\njohn_has_grade_Russian = john_grades['Russian'] != 'E'\n\ngrade_geology_follows_physics = john_grades['geology'] > john_grades['physics']\ngrade_physics_precedes_geology = john_grades['physics'] < john_grades['geology']\n\ngrade_Italian_follows_Russian = john_grades['Italian'] > john_grades['Russian']\ngrade_Russian_precedes_Italian = john_grades['Russian'] < john_grades['Italian']\n\ngrade_economics_higher_than_history = john_grades['economics'] > john_grades['history']\ngrade_

In [158]:
pprint(declarations)

('```python\n'
 "grades = EnumSort(['A', 'B', 'C', 'D', 'E'])\n"
 'consecutive_grades = Function([grades] -> [bool])\n'
 "john_grades = Function([['economics', 'geology', 'history', 'Italian', "
 "'physics', 'Russian']] -> [grades])\n"
 '\n'
 '# Conditions\n'
 "john_has_grade_economics = john_grades['economics'] != 'E'\n"
 "john_has_grade_geology = john_grades['geology'] != 'E'\n"
 "john_has_grade_history = john_grades['history'] != 'E'\n"
 "john_has_grade_Italian = john_grades['Italian'] != 'E'\n"
 "john_has_grade_physics = john_grades['physics'] != 'E'\n"
 "john_has_grade_Russian = john_grades['Russian'] != 'E'\n"
 '\n'
 "grade_geology_follows_physics = john_grades['geology'] > "
 "john_grades['physics']\n"
 "grade_physics_precedes_geology = john_grades['physics'] < "
 "john_grades['geology']\n"
 '\n'
 "grade_Italian_follows_Russian = john_grades['Italian'] > "
 "john_grades['Russian']\n"
 "grade_Russian_precedes_Italian = john_grades['Russian'] < "
 "john_grades['Italian']\n"
 '\n'


#### 直接转成 python z3

In [222]:
z3_declaration_template = """
## Instruction
To accurately translate the context into a first-order logical expression, please analyze and extract the Declaration from the context. Please DO NOT translate the constraints;

## Principle
1. Type Definition: For all entities of the same type, use the EnumSort() method to define a new type.
````code
NewType, (Entity_1, Entity_2, Entity_3, ... ) = EnumSort(TypeName, ('Entity_1', 'Entity_2', 'Entity_3', ... ))
````
- Entity_* : Entities of the same type mentioned in the text
- NewType: The name of the newly defined Z3 entity type
2. Constant
Constants are ordinary Python variables; they can also be of List type, where elements in the List are entity objects;
3. Function Definition: Functions generally represent predicate logic relationships in the text, so the number of inputs is one or more, and there is only one output type.
````code
FunctionName = Function(function_name , InputSort_1, InputSort_2,..., OutputSort)
````
Defining a function requires three formal parameters:
- function_name: The name of the function (predicate)
- **InputSort: One or more Z3 data types (including new types defined by us). If the function has multiple inputs, separate the multiple input types with commas;
- OutputSort: The output Z3 data type (including new types defined by us);
4. The constraints should not be included in the response!
5. Please follow the above z3 python code grammar!
6. The response should not contain any statements other than parts A, B, and C.


## Example
Please refer to the following example:

---

## Example 1:

### Input Text
Four boys—Fred, Juan, Marc, and Paul—and three girls—Nita, Rachel, and Trisha—will be assigned to a row of five adjacent lockers, numbered consecutively 1 through 5, arranged along a straight wall. The following conditions govern the assignment of lockers to the seven children: Each locker must be assigned to either one or two children, and each child must be assigned to exactly one locker. Each shared locker must be assigned to one girl and one boy. Juan must share a locker, but Rachel cannot share a locker. Nita's locker cannot be adjacent to Trisha's locker. Fred must be assigned to locker 3.",

### Declaration
#### Type Definition
ChildrenSort, (Fred, Juan, Marc, Paul, Nita, Rachel, Trisha) = EnumSort('Children', ['Fred', 'Juan', 'Marc', 'Paul', 'Nita', 'Rachel', 'Trisha'])
LockerSort, (locker_1, locker_2, locker_3, locker_4, locker_5) = EnumSort('Locker', [1, 2, 3, 4, 5])

#### Constant
children = [Fred, Juan, Marc, Paul, Nita, Rachel, Trisha]
lockers = [locker_1, locker_2, locker_3, locker_4, locker_5]

#### Function
assigned = Function('assigned', children, lockers) # means that [children]--assigned--> [lockers]
multi_assigned = Function('multi_assigned', children, children,  lockers) # means that [[children1], [children2]] --assigned--> [lockers]


---
## Example 2:

### Input Text
A music store carries exactly ten types of CDs—both new and used of each of jazz, opera, pop, rap, and soul. The store is having a sale on some of these types of CDs. The following conditions must apply: Used pop is on sale; new opera is not. If both types of pop are on sale, then all soul is. If both types of jazz are on sale, then no rap is. If neither type of jazz is on sale, then new pop is. If either type of rap is on sale, then no soul is.
### Declaration
#### Type Definition
MusicSort, (new_jazz, used_jazz, new_opera, used_opera, new_pop, used_pop, new_rap, used_rap, new_soul, used_soul) = EnumSort('Type', [new_jazz, used_jazz, new_opera, used_opera, new_pop, used_pop, new_rap, used_rap, new_soul, used_soul])
#### Function
on_sale = Function('on_sale', MusicSort, BoolSort())
#### Constant
music = [new_jazz, used_jazz, new_opera, used_opera, new_pop, used_pop, new_rap, used_rap, new_soul, used_soul]

---

Now, please extract the Declaration from the given text  and put the Declaration python code into ```python``` block:

### Input Text
"""

In [161]:
declarations_v2_res = llm.generate(data=z3_declaration_template+context,args=args)

In [162]:
pprint(declarations_v2_res)

('```python\n'
 "GradeSort, (A, B, C, D, E) = EnumSort('Grade', ['A', 'B', 'C', 'D', 'E'])\n"
 "consecutive_grades = Function('consecutive_grades', GradeSort, GradeSort)\n"
 '\n'
 '# Given constants\n'
 'grades = [A, B, C, D, E]\n'
 "john_grades = {A: 'economics', B: 'geology', C: 'history', D: 'Italian', E: "
 "'physics', F: 'Russian'}\n"
 "john_grades_in_order = {'economics': A, 'geology': B, 'history': C, "
 "'Italian': D, 'physics': E, 'Russian': F}\n"
 '\n'
 '# Constraints\n'
 'def grade_hierarchy(john_grade):\n'
 '    return john_grades_in_order[john_grade]\n'
 '\n'
 '# Relationships\n'
 'grade_economics_higher_than_history = '
 "Function('grade_economics_higher_than_history', GradeSort, BoolSort())\n"
 'grade_geology_higher_than_physics = '
 "Function('grade_geology_higher_than_physics', GradeSort, BoolSort())\n"
 'grade_physics_higher_than_italian = '
 "Function('grade_physics_higher_than_italian', GradeSort, BoolSort())\n"
 'grade_physics_not_equal_to_russian = '
 "Function('g

In [223]:
def extract_declarations(text):
    declarations_v2_res = llm.generate(data=z3_declaration_template+text,args=args)
    pattern = r"```python\s+(.*?)```"
    matches = re.findall(pattern, declarations_v2_res, flags=re.DOTALL)
    if matches:
        return matches[-1]
    else:
        return None

### Step 4: Rollout （开始尝试在 Step 里面翻译）

In [165]:
gen_template = r"""
You are a logical reasoning expert. Your task is to analyze the given problem step by step, and output your reasoning in a structured format.

## INSTRUCTIONS
1. **Reasoning Process**: Think through the problem step by step. Each step should be a clear logical progression.
2. **Premise List**: In every step, you must include a `<premise_list>` block that lists ALL currently available premises.
    The premise list include:
   - All premises given in the problem
   - All conclusions derived in previous steps
   - Any assumption explicitly introduced earlier
3. **Output Format**: Each reasoning step must be enclosed in `<step>` and `</step>` tags.
4. **Step Structure**: Within each step, use:
   - `<premise_list>` for the set of available premises
   - `<premise>` for the premises used in this step
   - `<conclusion>` for the conclusion of this step
5. **Step Separation**: Each complete step block must be separated by exactly TWO newlines (\n\n).
6. **Final Answer**: At the end, after all steps, provide the correct answer index within `\boxed{}`.

## OUTPUT FORMAT SPECIFICATION
Your output should have this exact structure:
<step>
<premise_list>
    <p1> premises 1 given in the problem </p1>
    <p2> Premises 2 given in the problem </p2>
    ...
    <c1> The derived intermediate conclusion 1 from previous steps</c1>
    <c2> The derived intermediate conclusion 2 from previous steps</c2>
    ...
</premise_list>
<premise>Premise used in this step</premise>
<premise>Another premise used in this step</premise>
<conclusion>Conclusion of this step</conclusion>
</step>

<step>
<premise_list>
    <p1> premises 1 given in the problem </p1>
    <p2> Premises 2 given in the problem </p2>
    ...
    <c1> The derived intermediate conclusion 1 from previous steps</c1>
    <c2> The derived intermediate conclusion 2 from previous steps</c2>
    ...
</premise_list>
<premise>Premise used in this step</premise>
<conclusion>Conclusion of this step</conclusion>
</step>

[More steps as needed...]

\boxed{answer_index}


## TAG RULES
- Each reasoning unit must be wrapped in `<step>` `</step>`
- Each step must contain EXACTLY ONE `<premise_list>` block, ONE '<conclusion>` block and at less one `<premise>` block
- `<premise_list>` must list ALL currently available premises
- `<premise>` contains only the premises used in THIS step
- `<conclusion>` contains only the new inference of THIS step
- Each step must have at least:
  - one `<premise_list>`
  - one `<premise>`
  - one `<conclusion>`
- The last step must contain the final logical conclusion


## PREMISE LIST RULES (IMPORTANT)

For step 1:
premise_list = all premises from the problem

For step N:
premise_list =  
- all original premises  
- all previous conclusions  

The premise_list must grow monotonically.


## STEP SEPARATION RULES

- Step 1
<step>...</step>

(blank line)

- Step 2
<step>...</step>

(blank line)

- Step 3
<step>...</step>


## EXAMPLE

Input:
"<p1>All cats are animals.</p1> \n\n <p2> Fluffy is a cat. What is Fluffy? </p2>"

Output:

<step>
<premise_list>
<p1>All cats are animals</p1>
<p2>Fluffy is a cat</p2>
</premise_list>
<premise>All cats are animals</premise>
<premise>Fluffy is a cat</premise>
<conclusion>Fluffy is an animal</conclusion>
</step>

<step>
<premise_list>
<p1>All cats are animals</p1>
<p2>Fluffy is a cat</p2>
<c1>Fluffy is an animal</c1>
</premise_list>
<premise>Fluffy is an animal</premise>
<conclusion>Therefore the answer is animal</conclusion>
</step>

\boxed{animal}


## COMMON MISTAKES TO AVOID

1. ❌ DO NOT Missing `<premise_list>`;
2. ❗️ Premise list must include previous conclusions;
3. ❌ DO NOT USE a premise not listed in premise list;
4. ❌ DO NOT include Multiple conclusions in one step;
5. ❌ DO NOT PUT reasoning after \boxed{}
6. ❌ Context can not be shrinking instead of growing


## SPECIAL CASES

- If no choices → output only steps
- If assumption needed → add it to context in next step
- If math → show calculation in premise


Now, solve the following problem.
"""

In [164]:
context = """
<p1>John receives one grade for each of the following six courses: economics, geology, history, Italian, physics, and Russian.</p1>
<p2>From highest to lowest, the possible grades are A, B, C, D, and E. E is the only failing grade.</p2>
<p3>Two letter grades are consecutive if and only if they are adjacent in the alphabet. </p3>
<p4>John's grades in geology and physics are consecutive. His grades in Italian and Russian are consecutive.</p4>
<p5>He receives a higher grade in economics than in history. </p5> 
<p6> He receives a higher grade in geology than in physics. </p6>
<p7> Assume that John's grade in physics is higher than his grade in Italian and consecutive with it and that his grades in Russian and physics differ</p7>
"""
hypothesis = """
<h1> (A)John receives both an A and a B. </h1>
<h2> (B)John receives both an A and a C. </h2>
<h3> (C)John receives both a B and a D. </h3>
<h4> (D)John receives both a B and a D. </h4>
<h5> (E) John receives both a D and an E. </h5>
"""
question = ". Which one of the following must be true?"
options = "(A)John receives both an A and a B.\n\n (B)John receives both an A and a C. \n\n (C)John receives both a B and a D. \n\n (D)John receives both a B and a D. \n\n (E) John receives both a D and an E."
gt = 'C'

In [166]:
gen_prompt = gen_template + context + question + hypothesis
gen_res = llm.generate(data=gen_prompt, args=args)
gen_res

"<step>\n<premise_list>\n    <p1>John receives one grade for each of the following six courses: economics, geology, history, Italian, physics, and Russian.</p1>\n    <p2>From highest to lowest, the possible grades are A, B, C, D, and E. E is the only failing grade.</p2>\n    <p3>Two letter grades are consecutive if and only if they are adjacent in the alphabet. </p3>\n    <p4>John's grades in geology and physics are consecutive. His grades in Italian and Russian are consecutive.</p4>\n    <p5>He receives a higher grade in economics than in history. </p5>\n    <p6> He receives a higher grade in geology than in physics. </p6>\n    <p7> Assume that John's grade in physics is higher than his grade in Italian and consecutive with it and that his grades in Russian and physics differ</p7>\n</premise_list>\n<premise>John's grades in geology and physics are consecutive.</premise>\n<premise>John's grades in Italian and Russian are consecutive.</premise>\n<premise>John's grade in physics is highe

In [167]:
pprint(gen_res)

('<step>\n'
 '<premise_list>\n'
 '    <p1>John receives one grade for each of the following six courses: '
 'economics, geology, history, Italian, physics, and Russian.</p1>\n'
 '    <p2>From highest to lowest, the possible grades are A, B, C, D, and E. E '
 'is the only failing grade.</p2>\n'
 '    <p3>Two letter grades are consecutive if and only if they are adjacent '
 'in the alphabet. </p3>\n'
 "    <p4>John's grades in geology and physics are consecutive. His grades in "
 'Italian and Russian are consecutive.</p4>\n'
 '    <p5>He receives a higher grade in economics than in history. </p5>\n'
 '    <p6> He receives a higher grade in geology than in physics. </p6>\n'
 "    <p7> Assume that John's grade in physics is higher than his grade in "
 'Italian and consecutive with it and that his grades in Russian and physics '
 'differ</p7>\n'
 '</premise_list>\n'
 "<premise>John's grades in geology and physics are consecutive.</premise>\n"
 "<premise>John's grades in Italian and Russian 

In [173]:
def get_step_list(text_content):
    Steps = []
    pattern = r"<step.*?>(.*?)</step>"
    matches = re.findall(pattern, text_content, flags=re.DOTALL)
    for i, content in enumerate(matches, 1):
        clean_content = content.strip()
        Steps.append(clean_content)
    return Steps

In [174]:
Steps = get_step_list(gen_res)

In [187]:
def get_premise_conclusion(step_content):
    premise_list = []
    pattern = r"<premise>(.*?)</premise>"
    matches = re.findall(pattern, step_content, flags=re.DOTALL)
    if matches:
        for i, content in enumerate(matches, 1):
            clean_content = content.strip()
            premise_list.append(clean_content)

    pattern = r"<conclusion>(.*?)</conclusion>"
    matches = re.findall(pattern, step_content, flags=re.DOTALL)
    conclusion = matches[-1] if matches else None
    return premise_list, conclusion

In [205]:
for i, step_content in enumerate(Steps):
    print("="*10 + f"Step {i}" + "="*10)
    premise_list, conclusion = get_premise_conclusion(step_content)
    print(premise_list)
    print(conclusion)
    premises = " ".join(premise_list)
    text = premises + conclusion
    declaration = extract_declarations(text)
    print(declaration)

==========Step 0==========
["John's grades in geology and physics are consecutive.", "John's grades in Italian and Russian are consecutive.", "John's grade in physics is higher than his grade in Italian and consecutive with it.", 'His grades in Russian and physics differ.']
John's grade in physics is either A or B, and his grade in Italian is either B or A.
None
==========Step 1==========
["John's grades in geology and physics are consecutive.", "John's grades in Italian and Russian are consecutive.", "John's grade in physics is higher than his grade in Italian and consecutive with it.", 'His grades in Russian and physics differ.']
John cannot receive an E grade in any course.
GradeSort, (geology_grade, physics_grade, italian_grade, russian_grade) = EnumSort('Grade', ['geology_grade', 'physics_grade', 'italian_grade', 'russian_grade'])

cannot_receive_e_grade = Function('cannot_receive_e_grade', GradeSort, BoolSort())

# John's grades in geology and physics are consecutive.
consecutive

对于 Function 的翻译依旧翻译不好，主要反映在这几个方面：
1. 要求 LLM 不要给出 Constraints 但它依旧会给出对应 Contraints，而且 contraints 依旧做不对
2. 格式不对有错；

尝试先把 premises 变成 atom declaration，再抽取 declaration

In [225]:
for i, step_content in enumerate(Steps):
    print("="*10 + f"Step {i}" + "="*10)
    premise_list, conclusion = get_premise_conclusion(step_content)
    print("## PREMISE ##")
    print(premise_list)
    print("## CONCLUSION ##")
    print(conclusion)
    premises = " ".join(premise_list)
    
    premise_atom = extract_atom(atomic_template, premises, llm, args)
    conclusion_atom = extract_atom(atomic_template, conclusion, llm, args)

    pattern = pattern = r"Atom\s+\d+:\s+(.*)"
    pre_matches = re.findall(pattern, premise_atom)
    pre_atom_list = [m.strip() for m in pre_matches]

    cls_matches = re.findall(pattern, conclusion_atom)
    cls_atom_list = [m.strip() for m in cls_matches]

    atom_list_total = pre_atom_list + cls_atom_list
    print("## Premise Atom ##")
    print(pre_atom_list)
    print("## Conclusion Atom ##")
    print(cls_atom_list)
    declaration = extract_declarations(" ".join(atom_list_total))
    print("-"*100)
    print(declaration)
    

==========Step 0==========
## PREMISE ##
["John's grades in geology and physics are consecutive.", "John's grades in Italian and Russian are consecutive.", "John's grade in physics is higher than his grade in Italian and consecutive with it.", 'His grades in Russian and physics differ.']
## CONCLUSION ##
John's grade in physics is either A or B, and his grade in Italian is either B or A.
## Premise Atom ##
["John's grades in geology and physics are consecutive.", "John's grades in Italian and Russian are consecutive.", "John's grade in physics is higher than his grade in Italian.", "John's grade in physics is consecutive with his grade in Italian.", 'The grades in Russian and physics differ.']
## Conclusion Atom ##
["John's grade in physics is A or John's grade in physics is B", "John's grade in Italian is B or John's grade in Italian is A"]
----------------------------------------------------------------------------------------------------
None
==========Step 1==========
## PREMISE ##

更换为 Qwen2.5-7B 的效果

In [227]:
for i, step_content in enumerate(Steps):
    print("="*10 + f"Step {i}" + "="*10)
    premise_list, conclusion = get_premise_conclusion(step_content)
    print("## PREMISE ##")
    print(premise_list)
    print("## CONCLUSION ##")
    print(conclusion)
    premises = " ".join(premise_list)
    
    premise_atom = extract_atom(atomic_template, premises, llm, args)
    conclusion_atom = extract_atom(atomic_template, conclusion, llm, args)

    pattern = pattern = r"Atom\s+\d+:\s+(.*)"
    pre_matches = re.findall(pattern, premise_atom)
    pre_atom_list = [m.strip() for m in pre_matches]

    cls_matches = re.findall(pattern, conclusion_atom)
    cls_atom_list = [m.strip() for m in cls_matches]

    atom_list_total = pre_atom_list + cls_atom_list
    print("## Premise Atom ##")
    print(pre_atom_list)
    print("## Conclusion Atom ##")
    print(cls_atom_list)
    declaration = extract_declarations(" ".join(atom_list_total))
    print("-"*100)
    print(declaration)
    

==========Step 0==========
## PREMISE ##
["John's grades in geology and physics are consecutive.", "John's grades in Italian and Russian are consecutive.", "John's grade in physics is higher than his grade in Italian and consecutive with it.", 'His grades in Russian and physics differ.']
## CONCLUSION ##
John's grade in physics is either A or B, and his grade in Italian is either B or A.
## Premise Atom ##
["John's grade in geology is consecutive with his grade in physics.", "John's grade in physics is higher than his grade in Italian.", "John's grade in Italian is consecutive with his grade in Russian.", "John's grade in physics is consecutive with his grade in Russian.", "John's grade in Russian is not consecutive with his grade in physics.", 'John has grades in geology, physics, Italian, and Russian.']
## Conclusion Atom ##
["John's grade in physics is A.", "John's grade in physics is B.", "John's grade in Italian is B.", "John's grade in Italian is A."]
----------------------------

### Step 4: Translation（看着每一步的 declaration 语法稍微对了一点，尝试开始翻译表达式）

In [239]:
trans_template = """
### Instruction
You are an expert in formal logic and knowledge representation. Translate natural language sentences into precise first-order logic (FOL) formulas (python z3 code) based on the provided type declarations and function signatures.

## Task Specification
Given declarations and a sentence, produce the correct FOL translation following these strict rules:

### 1. FOL Syntax Requirements
- Use the exact identifiers from declarations (case-sensitive)
- Quantifiers: `ForAll([var:type], formula)` or `Exists([var:type], formula)`
- Logical connectives: `And(f1, f2, ...)`, `Or(f1, f2, ...)`, `Not(formula)`, `Implies(f1, f2)`
- Equality/inequality: `t1 == t2` or `t1 != t2`
- Special predicates: 
  - `Distinct([t1, t2, ...])` for "all arguments are distinct"
  - `Count([x:s], formula) >= n` for cardinality constraints
  - `t1 IN enum_sort_name` for membership (alternative to `t1 == elem1 Or t1 == elem2 ...`)

### 2. Translation Principles
- **Faithfulness**: Preserve the exact meaning, no simplifications
- **Exhaustiveness**: Handle all quantifiers explicitly
- **Type safety**: All variables must have declared types
- **Function application**: `f(x, y)` not `f x y`
- **No abbreviations**: Write out full formulas and each constraint must be written in the form of a single line.
- **Please do not use the same letter for different types of variables**: for example, ForAll([p:person], ...), and ForAll([p:location],...). Therefore, I will provide the already generated FOL expressions at the same time; when using universal quantifiers and variables, please refer to the existing FOL expressions and do not use the same letter for different types of variables.
- **The expression should be put into the "s.add()", such as ```s.add(ForAll([m], eats(Vladimir, m) != eats(Wendy, m)))``` and the variable 'm' should be clarify before the add operation (such as ```m = Const('m',meals)```)

### 3. Examples Pattern
# Declarations 
people, (Vladimir, Wendy) = EnumSort('people',['Vladimir', 'Wendy']) 
meals, (breakfast, lunch, dinner, snack) = EnumSort('meals', ['breakfast', 'lunch', 'dinner', 'snack']) 
foods, (fish, hot_cakes, macaroni, omelet, poached_eggs) = EnumSort('foods', ['fish', 'hot_cakes', 'macaroni', 'omelet', 'poached_eggs']) 
eats = Function('eat', people, meals, foods) # [people, meals] -> [foods]
m = Const('m',meals)
p = Const('p',people)


# Sentence 
At no meal does Vladimir eat the same kind of food as Wendy 
# FOL expression
s.add(ForAll([m], eats(Vladimir, m) != eats(Wendy, m)))

# Sentence 
Neither of them eats the same kind of food more than once during the day 
# FOL expression
s.add(ForAll([p], Distinct([m], eats(p, m))))

# Sentence 
For breakfast, each eats exactly one of the following: hot cakes, poached eggs, or omelets 
# FOL expression 
s.add(ForAll([p], Or(eats(p, breakfast) == hot_cakes, eats(p, breakfast) == poached_eggs, eats(p, breakfast) == omelet))) 

--- 

Please translate the following sentences to first order logic program and put the program into ```code``` block:

## Declarations
${declarations}

## Sentence
${sentence}
"""
from string import Template
trans_T = Template(trans_template)

In [240]:
test_sentence = "John's grade in geology is consecutive with his grade in physics."
declaration = r"""
#### Type Definition
StudentSort, (John) = EnumSort('Student', ['John'])

CourseSort, (Geology, Physics, Italian, Russian) = EnumSort('Course', ['Geology', 'Physics', 'Italian', 'Russian'])

GradeSort = EnumSort('Grade', ['A', 'B', 'C', 'D', 'F'])

#### Constant
john = John

courses = [Geology, Physics, Italian, Russian]

#### Function
grade_in = Function('grade_in', StudentSort, CourseSort, GradeSort)

#### Variable
s = Const('s', StudentSort)
c = Const('c', CourseSort)
g = Const('g', GradeSort)

"""


trans_prompt = trans_T.safe_substitute(declarations=declaration, sentence=test_sentence)
trans_prompt

'\n### Instruction\nYou are an expert in formal logic and knowledge representation. Translate natural language sentences into precise first-order logic (FOL) formulas (python z3 code) based on the provided type declarations and function signatures.\n\n## Task Specification\nGiven declarations and a sentence, produce the correct FOL translation following these strict rules:\n\n### 1. FOL Syntax Requirements\n- Use the exact identifiers from declarations (case-sensitive)\n- Quantifiers: `ForAll([var:type], formula)` or `Exists([var:type], formula)`\n- Logical connectives: `And(f1, f2, ...)`, `Or(f1, f2, ...)`, `Not(formula)`, `Implies(f1, f2)`\n- Equality/inequality: `t1 == t2` or `t1 != t2`\n- Special predicates: \n  - `Distinct([t1, t2, ...])` for "all arguments are distinct"\n  - `Count([x:s], formula) >= n` for cardinality constraints\n  - `t1 IN enum_sort_name` for membership (alternative to `t1 == elem1 Or t1 == elem2 ...`)\n\n### 2. Translation Principles\n- **Faithfulness**: Pres

In [241]:
trans_res = llm.generate(data=trans_prompt, args=args)

In [ ]:
pattern = "```python(.?*) ```"

In [242]:
pprint(trans_res)

('```python\n'
 "s = Const('s', StudentSort)\n"
 "c = Const('c', CourseSort)\n"
 "g = Const('g', GradeSort)\n"
 '\n'
 "# John's grade in geology is consecutive with his grade in physics.\n"
 's.add(And(grade_in(John, Geology) == g, grade_in(John, Physics) == If(g == '
 "'A', 'B', If(g == 'B', 'C', If(g == 'C', 'D', 'F')))))\n"
 '```')


In [276]:
import sys
from io import StringIO

# CAN: 假设这是 LLM 返回的代码字符串
llm_generated_code = """
x = 10
y = 20
result = x * y
print(f"Calculated result: {result}")
for i in range(3):
    print(f"Loop {i}")
"""

def execute_and_capture_output(code_string):
    # 1. 创建一个类似文件的对象来捕获输出
    old_stdout = sys.stdout
    redirected_output = StringIO()
    sys.stdout = redirected_output  # 重定向标准输出

    try:
        # 2. 核心：执行字符串代码
        # globals() 传递当前的全局变量，这样代码可以使用导入的库
        exec(code_string, globals()) 
    except Exception as e:
        print(f"执行出错: {e}")
    finally:
        # 3. 恢复标准输出，否则后面的 print 你都看不到了！
        sys.stdout = old_stdout
        # return None

    # 4. 获取捕获到的内容
    return redirected_output.getvalue()

# CAN: 运行并查看结果
output = execute_and_capture_output(llm_generated_code)

print("CAN: 以下是自动执行的结果：")
print("-" * 30)
print(output)
print("-" * 30)


CAN: 以下是自动执行的结果：
------------------------------
Calculated result: 200
Loop 0
Loop 1
Loop 2

------------------------------


In [282]:
def wrap_z3_code(declaration, expression):
    z3_code = "# Auto-generated Z3 code\n\n"
    z3_code += "from z3 import *\n\n"
    z3_code += "s = Solver()\n\n"
    z3_code += "s.reset()"
    z3_code += "# --- Declarations ---\n\n"
    z3_code += declaration + "\n\n"
    z3_code += "# --- Expressions ---\n\n"
    z3_code += expression + "\n\n"
    z3_code += "result = s.check()\n"
    z3_code += "print(f'Result: {result}')\n"
    z3_code += "if result == sat:\n"
    z3_code += "    print('Model:', s.model())\n"
    return z3_code

In [278]:
expression = ""
code = wrap_z3_code(declaration,expression)
out = execute_and_capture_output(code)
out

"执行出错: b'enumeration sort name is already declared'\n"

In [272]:
pprint(code)

('# Auto-generated Z3 code\n'
 '\n'
 'from z3 import *\n'
 '\n'
 's = Solver()\n'
 '\n'
 's.reset()# --- Declarations ---\n'
 '\n'
 '\n'
 '#### Type Definition\n'
 "StudentSort, (John) = EnumSort('Student', ['John'])\n"
 '\n'
 "CourseSort, (Geology, Physics, Italian, Russian) = EnumSort('Course', "
 "['Geology', 'Physics', 'Italian', 'Russian'])\n"
 '\n'
 "GradeSort = EnumSort('Grade', ['A', 'B', 'C', 'D', 'F'])\n"
 '\n'
 '#### Constant\n'
 'john = John\n'
 '\n'
 'courses = [Geology, Physics, Italian, Russian]\n'
 '\n'
 '#### Function\n'
 "grade_in = Function('grade_in', StudentSort, CourseSort, GradeSort)\n"
 '\n'
 '#### Variable\n'
 "s = Const('s', StudentSort)\n"
 "c = Const('c', CourseSort)\n"
 "g = Const('g', GradeSort)\n"
 '\n'
 '\n'
 '\n'
 '# --- Expressions ---\n'
 '\n'
 '\n'
 '\n'
 'result = s.check()\n'
 "print(f'Result: {result}')\n"
 'if result == sat:\n'
 "    print('Model:', s.model())\n")


### Step 5: Code Refinement 尝试直接让 LLM 接收 Python 环境的反馈自己优化代码

In [279]:
def correct_z3_code(code, error):
    fix_bug_prompt = """
    You are an experienced Python programmer who excels at fixing bugs in code. Given a piece of Python code regarding the use of Z3 solver to verify logical correctness and its error message, please comprehensively judge what went wrong with this code, and place the fixed, syntactically correct, and executable Python code into a ```python ``` block
    ### Given Python Code
    ${code}
    
    ### Error Message
    ${error}
    
    """
    from string import Template
    fix_T = Template(fix_bug_prompt)
    fix_prompt = fix_T.safe_substitute(code=code, error=error)
    fix_out = llm.generate(data=fix_prompt, args=args)
    # print(fix_out)
    pattern = r"```python\s+(.*?)```"
    matches = re.findall(pattern, fix_out, re.DOTALL)
    return matches[-1]

def correct_loop(code,out):
    while "执行出错" in out:
        code = correct_z3_code(code,out)
        # print(new_code)
        out = execute_and_capture_output(code)
        print(out)

correct_loop(code,out)

执行出错: b'enumeration sort name is already declared'

执行出错: b'enumeration sort name is already declared'

执行出错: b'enumeration sort name is already declared'

执行出错: b'enumeration sort name is already declared'

执行出错: b'enumeration sort name is already declared'

执行出错: b'enumeration sort name is already declared'

执行出错: b'enumeration sort name is already declared'

执行出错: b'enumeration sort name is already declared'

执行出错: b'enumeration sort name is already declared'



KeyboardInterrupt: 

是不是报错信息太少，所以无法定位到错误的具体位置，如果给出更详细的 TB 信息呢？

In [285]:
import subprocess
import sys

def run_code_in_subprocess(code_string):
    # print("CAN: 正在子进程中启动代码...")
    
    try:
        # 使用当前 python 解释器 (sys.executable) 执行代码
        # capture_output=True 会同时捕获 stdout 和 stderr
        # text=True 会让结果以字符串形式返回，而不是字节
        result = subprocess.run(
            [sys.executable, "-c", code_string],
            capture_output=True,
            text=True,
            timeout=10 # 设置超时防止死循环
        )
        
        if result.returncode == 0:
            return {
                "success": True,
                "output": result.stdout,
                "error": None
            }
        else:
            # 如果 returncode != 0，说明出错了
            # stderr 通常包含 Traceback
            return {
                "success": False,
                "output": result.stdout,
                "error": result.stderr 
            }
            
    except subprocess.TimeoutExpired:
        return {"success": False, "error": "CAN: 代码执行超时！"}
    except Exception as e:
        return {"success": False, "error": str(e)}

# 测试代码
dangerous_code = """
import sys
def foo():
    bar()
def bar():
    raise ValueError("Deep failure inside subprocess!")
foo()
"""

res = run_code_in_subprocess(dangerous_code)
if not res["success"]:
    # print("\nCAN: 子进程报错捕获成功：")
    print(res["error"])


Traceback (most recent call last):
  File "<string>", line 7, in <module>
  File "<string>", line 4, in foo
  File "<string>", line 6, in bar
ValueError: Deep failure inside subprocess!



In [286]:
expression = ""
code = wrap_z3_code(declaration,expression)
out = execute_and_capture_output(code)
out


def correct_z3_code(code, error):
    fix_bug_prompt = """
    You are an experienced Python programmer who excels at fixing bugs in code. Given a piece of Python code regarding the use of Z3 solver to verify logical correctness and its error message, please comprehensively judge what went wrong with this code, and place the fixed, syntactically correct, and executable Python code into a ```python ``` block
    ### Given Python Code
    ${code}
    
    ### Error Message
    ${error}
    
    """
    from string import Template
    fix_T = Template(fix_bug_prompt)
    fix_prompt = fix_T.safe_substitute(code=code, error=error)
    fix_out = llm.generate(data=fix_prompt, args=args)
    # print(fix_out)
    pattern = r"```python\s+(.*?)```"
    matches = re.findall(pattern, fix_out, re.DOTALL)
    return matches[-1]

def correct_loop(code,res):
    code = wrap_z3_code(declaration,expression)
    res = run_code_in_subprocess(code)
    while not res["success"]:
        code = correct_z3_code(code,res["error"])
        # print(new_code)
        res = run_code_in_subprocess(code)
        print(res)

correct_loop(code,out)

{'success': False, 'output': '', 'error': 'Traceback (most recent call last):\n  File "<string>", line 16, in <module>\n  File "/home/chenzhb/.conda/envs/verl/lib/python3.10/site-packages/z3/z3.py", line 916, in Function\n    _z3_assert(is_sort(rng), "Z3 sort expected")\n  File "/home/chenzhb/.conda/envs/verl/lib/python3.10/site-packages/z3/z3.py", line 115, in _z3_assert\n    raise Z3Exception(msg)\nz3.z3types.Z3Exception: Z3 sort expected\n'}
{'success': False, 'output': '', 'error': 'Traceback (most recent call last):\n  File "<string>", line 16, in <module>\n  File "/home/chenzhb/.conda/envs/verl/lib/python3.10/site-packages/z3/z3.py", line 916, in Function\n    _z3_assert(is_sort(rng), "Z3 sort expected")\n  File "/home/chenzhb/.conda/envs/verl/lib/python3.10/site-packages/z3/z3.py", line 115, in _z3_assert\n    raise Z3Exception(msg)\nz3.z3types.Z3Exception: Z3 sort expected\n'}
{'success': False, 'output': '', 'error': 'Traceback (most recent call last):\n  File "<string>", line

KeyboardInterrupt: 

好吧，没辙了

#### 整段进行翻译
因为在单句进行翻译的时候，会重复给出变量的定义，这回引发重复定义的 error，也会增加出错的概率，所以，如果我将每个 step 的所有 premise 绑在一起翻译是否会好一点？

In [288]:
sentence= "John's grades in geology and physics are consecutive. John's grades in Italian and Russian are consecutive. John's grade in physics is higher than his grade in Italian and consecutive with it. His grades in Russian and physics differ."
trans_prompt = trans_T.safe_substitute(declarations=declaration, sentence=test_sentence)
trans_res = llm.generate(data=trans_prompt, args=args)
pattern = r"```python\s+(.*?)```"
code = re.findall(pattern, trans_res, re.DOTALL)[-1]
code

"s = Const('s', StudentSort)\nc = Const('c', CourseSort)\ng = Const('g', GradeSort)\n\n# Translate the sentence: John's grade in geology is consecutive with his grade in physics.\n# The grades are consecutive if the difference between them is exactly 1.\n# We assume the grades are ordered as follows: A > B > C > D > F\ns.add(And(grade_in(john, Geology) == g,\n          grade_in(john, Physics) == If(g == 'A', 'B', \n                                         If(g == 'B', 'C', \n                                            If(g == 'C', 'D', \n                                               If(g == 'D', 'F', 'A'))))))\n"

In [290]:
z3_ = wrap_z3_code(declaration, code)
pprint(z3_)

('# Auto-generated Z3 code\n'
 '\n'
 'from z3 import *\n'
 '\n'
 's = Solver()\n'
 '\n'
 's.reset()# --- Declarations ---\n'
 '\n'
 '\n'
 '#### Type Definition\n'
 "StudentSort, (John) = EnumSort('Student', ['John'])\n"
 '\n'
 "CourseSort, (Geology, Physics, Italian, Russian) = EnumSort('Course', "
 "['Geology', 'Physics', 'Italian', 'Russian'])\n"
 '\n'
 "GradeSort = EnumSort('Grade', ['A', 'B', 'C', 'D', 'F'])\n"
 '\n'
 '#### Constant\n'
 'john = John\n'
 '\n'
 'courses = [Geology, Physics, Italian, Russian]\n'
 '\n'
 '#### Function\n'
 "grade_in = Function('grade_in', StudentSort, CourseSort, GradeSort)\n"
 '\n'
 '#### Variable\n'
 "s = Const('s', StudentSort)\n"
 "c = Const('c', CourseSort)\n"
 "g = Const('g', GradeSort)\n"
 '\n'
 '\n'
 '\n'
 '# --- Expressions ---\n'
 '\n'
 "s = Const('s', StudentSort)\n"
 "c = Const('c', CourseSort)\n"
 "g = Const('g', GradeSort)\n"
 '\n'
 "# Translate the sentence: John's grade in geology is consecutive with his "
 'grade in physics.\n'
 '# The 

In [291]:
out = execute_and_capture_output(code)

In [292]:
out

'执行出错: Z3 sort expected\n'

#### 7B 直接从题目抽 declaration

In [293]:
coref_output

"<p1>John receives one grade for each of the following six courses: economics, geology, history, Italian, physics, and Russian.</p1>\n<p2>From highest to lowest, the possible grades are A, B, C, D, and E. E is the only failing grade.</p2>\n<p3>Two letter grades are consecutive if and only if they are adjacent in the alphabet. </p3>\n<p4>John's grades in geology and physics are consecutive. John's grades in Italian and Russian are consecutive.</p4>\n<p5>John receives a higher grade in economics than in history. </p5>\n<p6> John receives a higher grade in geology than in physics. </p6>\n<p7> Assume that John's grade in physics is higher than his grade in Italian and consecutive with it and that his grades in Russian and physics differ</p7>"

In [297]:
declaration_prompt = """
To accurately translate the context into a first-order logical expression, please analyze and extract the Declaration from the context. Please refer to the following example:

---

## Input
Four boys—Fred, Juan, Marc, and Paul—and three girls—Nita, Rachel, and Trisha—will be assigned to a row of five adjacent lockers, numbered consecutively 1 through 5, arranged along a straight wall. The following conditions govern the assignment of lockers to the seven children: Each locker must be assigned to either one or two children, and each child must be assigned to exactly one locker. Each shared locker must be assigned to one girl and one boy. Juan must share a locker, but Rachel cannot share a locker. Nita's locker cannot be adjacent to Trisha's locker. Fred must be assigned to locker 3.",

## Output
children = EnumSort([Fred, Juan, Marc, Paul, Nita, Rachel, Trisha])
lockers = EnumSort([1, 2, 3, 4, 5])
assigned = Function([children] -> [lockers])

---
## Input
A music store carries exactly ten types of CDs—both new and used of each of jazz, opera, pop, rap, and soul. The store is having a sale on some of these types of CDs. The following conditions must apply: Used pop is on sale; new opera is not. If both types of pop are on sale, then all soul is. If both types of jazz are on sale, then no rap is. If neither type of jazz is on sale, then new pop is. If either type of rap is on sale, then no soul is.

## Output
types = EnumSort([new_jazz, used_jazz, new_opera, used_opera, new_pop, used_pop, new_rap, used_rap, new_soul, used_soul])
on_sale = Function([types] -> [bool])

---
Now, please extract the Declaration from the following problem  and put the Declaration into ```python``` block :

## Input
${context}

"""
dec_out = declaration_ext(declaration_prompt, coref_output)

In [299]:
pprint(dec_out)

('courses = EnumSort(["economics", "geology", "history", "Italian", "physics", '
 '"Russian"])\n'
 'grades = EnumSort(["A", "B", "C", "D", "E"])\n'
 'letter_consecutive = Function([(grades, grades)] -> [bool])\n'
 'grades_received = Function([courses] -> [grades])\n'
 '\n'
 '# Conditions\n'
 'letter_consecutive(grades("A"), grades("B")) == True\n'
 'letter_consecutive(grades("B"), grades("C")) == True\n'
 'letter_consecutive(grades("C"), grades("D")) == True\n'
 'letter_consecutive(grades("D"), grades("E")) == True\n'
 '\n'
 '# Geology and Physics grades are consecutive\n'
 'letter_consecutive(grades_received(courses("geology")), '
 'grades_received(courses("physics"))) == True\n'
 '\n'
 '# Italian and Russian grades are consecutive\n'
 'letter_consecutive(grades_received(courses("Italian")), '
 'grades_received(courses("Russian"))) == True\n'
 '\n'
 '# Economics grade is higher than History grade\n'
 'grades_received(courses("economics")) > grades_received(courses("history"))\n'
 '\n'

In [309]:
from fol_to_python_converter import convert_fol_problem, convert_and_execute_fol

declaration = """
courses = EnumSort([economics, geology, history, Italian, physics, Russian])
grades = EnumSort([A, B, C, D, E])
letter_consecutive = Function([(grades, grades)] -> [bool])
grades_received = Function([courses] -> [grades])


"""
constraints = """
# Conditions
letter_consecutive(grades("A"), grades("B")) == True
letter_consecutive(grades("B"), grades("C")) == True
letter_consecutive(grades("C"), grades("D")) == True
letter_consecutive(grades("D"), grades("E")) == True

# Geology and Physics grades are consecutive
letter_consecutive(grades_received(courses("geology")), grades_received(courses("physics"))) == True

# Italian and Russian grades are consecutive
letter_consecutive(grades_received(courses("Italian")), grades_received(courses("Russian"))) == True

# Economics grade is higher than History grade
grades_received(courses("economics")) > grades_received(courses("history"))

# Geology grade is higher than Physics grade
grades_received(courses("geology")) > grades_received(courses("physics"))

# Physics grade is higher than Italian grade and consecutive with it
grades_received(courses("physics")) > grades_received(courses("Italian"))
letter_consecutive(grades_received(courses("physics")), grades_received(courses("Italian"))) == True

# Grades in Russian and Physics differ
grades_received(courses("Russian")) != grades_received(courses("physics"))
"""
python_code = convert_fol_problem(declarations, constraints, output_file='./generated_problem.py')

Code have been save to the path: ./generated_problem.py


In [310]:
!python generated_problem.py

  File "/home/chenzhb/Workspaces/verl/mcts_utils/generated_problem.py", line 6
    john_grades = Function('john_grades', ['economics'_sort, 'geology'_sort, 'history'_sort, 'Italian'_sort, 'physics'_sort, 'Russian'_sort, grades_sort)
                                                                                                                                                        ^
SyntaxError: closing parenthesis ')' does not match opening parenthesis '['


直接生成的代码

In [311]:
from z3 import *

grades_sort, (A, B, C, D, E) = EnumSort('grades', ['A', 'B', 'C', 'D', 'E'])
grades = [A, B, C, D, E]
consecutive_grades = Function('consecutive_grades', grades_sort, BoolSort())
john_grades = Function('john_grades', ['economics', 'geology', 'history', 'Italian', 'physics', 'Russian', 'grades'])

# Constraints
solver = Solver()

solver.add(letter_consecutive(grades("A"), grades("B")) == True)
solver.add(letter_consecutive(grades("B"), grades("C")) == True)
solver.add(letter_consecutive(grades("C"), grades("D")) == True)
solver.add(letter_consecutive(grades("D"), grades("E")) == True)
solver.add(letter_consecutive(grades_received(courses("geology")), grades_received(courses("physics"))) == True)
solver.add(letter_consecutive(grades_received(courses("Italian")), grades_received(courses("Russian"))) == True)
solver.add(grades_received(courses("economics")) > grades_received(courses("history")))
solver.add(grades_received(courses("geology")) > grades_received(courses("physics")))
solver.add(grades_received(courses("physics")) > grades_received(courses("Italian")))
solver.add(letter_consecutive(grades_received(courses("physics")), grades_received(courses("Italian"))) == True)
solver.add(grades_received(courses("Russian")) != grades_received(courses("physics")))

# Check satisfiability
if solver.check() == sat:
    m = solver.model()
    print(m)
    print(1)
else:
    print('UNSAT')
    print(0)

Z3Exception: b'enumeration sort name is already declared'

人工修改后

In [ ]:
from z3 import *


grades_sort, (A, B, C, D, E) = EnumSort('grades', ['A', 'B', 'C', 'D', 'E'])
courses_sort, (economics, geology, history, Italian, physics, Russian) = EnumSort(
    'courses', ['economics', 'geology', 'history', 'Italian', 'physics', 'Russian']
)# 缺少类型定义
grades = [A, B, C, D, E]
courses = [economics, geology, history, Italian, physics, Russian]

letter_consecutive = Function('letter_consecutive', grades_sort, grades_sort, BoolSort())
john_grades = Function('john_grades', courses_sort, grades_sort)
grade_rank = Function('grade_rank', grades_sort, IntSort()) # 新增

# Constraints
solver = Solver()
solver.reset()

# 新增
solver.add(grade_rank(A) == 5)
solver.add(grade_rank(B) == 4)
solver.add(grade_rank(C) == 3)
solver.add(grade_rank(D) == 2)
solver.add(grade_rank(E) == 1)

solver.add(letter_consecutive(A, B) == True)
solver.add(letter_consecutive(B, C) == True)
solver.add(letter_consecutive(C, D) == True)
solver.add(letter_consecutive(D, E) == True)

solver.add(letter_consecutive(john_grades(geology), john_grades(physics)) == True)
solver.add(letter_consecutive(john_grades(Italian), john_grades(Russian)) == True)
solver.add(grade_rank(john_grades(economics)) > grade_rank(john_grades(history)))
solver.add(grade_rank(john_grades(geology)) > grade_rank(john_grades(physics)))
solver.add(grade_rank(john_grades(physics)) > grade_rank(john_grades(Italian)))
solver.add(letter_consecutive(john_grades(physics), john_grades(Italian)) == True)
solver.add(john_grades(Russian) != john_grades(physics))

# Check satisfiability
if solver.check() == sat:
    m = solver.model()
    print(m)
    print(1)
else:
    print('UNSAT')
    print(0)